# Exploratory Data Analysis & Statistical Analysis

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import statsmodels.api as sm

In [ ]:
BG_COLOR = "#23272e"  
BOX_COLOR = "#61afef"  
TEXT_COLOR = "#abb2bf"  

plt.rcParams.update(
    {
        "figure.facecolor": BG_COLOR,  
        "axes.facecolor": BG_COLOR, 
        "axes.edgecolor": "#3f444a",  
        "axes.labelcolor": TEXT_COLOR,  
        "text.color": TEXT_COLOR,  
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,  
        "grid.color": "#2c313c",  
    }
)

sns.set_theme(
    style="dark",
    rc={
        "figure.facecolor": BG_COLOR,
        "axes.facecolor": BG_COLOR,
        "text.color": "#abb2bf",
        "axes.labelcolor": "#abb2bf",
        "xtick.color": "#abb2bf",
        "ytick.color": "#abb2bf",
    },
)

In [ ]:
df = pd.read_parquet("../data/jev/jev_output_df.parquet")
df.info()

## Initial Pairplot

In [ ]:
pairplot_df = df.drop(columns=['text', 'content_type', 'rhetorical_target'])
sns.pairplot(pairplot_df)
plt.show()

## Correlation Heatmap

In [ ]:
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

## VIF Multicollinearity Test

In [ ]:
X_const = add_constant(pairplot_df)

vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_const.values, i) 
    for i in range(X_const.shape[1])
]

print(vif_data)

## Multiple Regression

In [ ]:
df_encoded = pd.get_dummies(df, columns=['content_type', 'rhetorical_target'], drop_first=True, dtype=int)

X = df_encoded.drop(columns=['ragebait', 'text'])
y = df['ragebait']
X = sm.add_constant(X)

model_sm = sm.OLS(y, X).fit()
print(model_sm.summary())